# Assignment 2: From Trees to Neural Networks

**Dataset:** UCI Bank Marketing (`bank-additional-full.csv`)

**Current scope:** Step 0 scaffold — setup, Colab data upload, and initial data load. Completed Step 1 is in `assign2-bank_step1.ipynb`.

> ## How to use this notebook
> Same correct implementation as the final/submission notebook, plus learning notes, TODOs, self-check questions, and report prompts.
> Open this file in Colab the same way: upload the `.ipynb`, then upload `bank-additional-full.csv` into the working directory.
> This learning notebook is for study and report prep; the **final** notebook is the Colab submission artifact.


> **Learning note — dual notebooks**
> - **Final notebook** = clean, submission-style code + professional Markdown (what graders open in Colab).
> - **Learning notebook** = same code + explanations you can study from and pull into the PDF.
> Keep them in sync for each step. Both expect `bank-additional-full.csv` uploaded into the Colab working directory.


## 0. Setup


> **Learning note — why fix `RANDOM_STATE`?**
> Reproducibility: the same split and sampling decisions must be recoverable. The handout requires `random_state=42` for the stratified split.


In [ ]:
import platform

import numpy as np
import pandas as pd

RANDOM_STATE = 42
DATA_FILE = "bank-additional-full.csv"

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"RANDOM_STATE = {RANDOM_STATE}")


### 0.1 Load the data

Place `bank-additional-full.csv` in the **same working directory as this notebook** (in Colab: upload via the Files sidebar).

Source: UCI Bank Marketing — extract `bank-additional-full.csv` from `bank-additional.zip`. The file is semicolon-separated.


> **TODO (learning)**
> - [ ] Upload `bank-additional-full.csv` to the Colab working directory before running
> - [ ] Record the exact file source (UCI Bank Marketing / `bank-additional.zip`) for the report Introduction


In [ ]:
from pathlib import Path

DATA_PATH = Path(DATA_FILE)
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_FILE} not found in the working directory. "
        "In Colab: Files → Upload → select bank-additional-full.csv, then re-run this cell."
    )

print(f"Found: {DATA_PATH.resolve()}")


## 1. Data loading and initial audit


> **Learning note — why audit before modeling?**
> Step 1 of the handout requires row/column counts, feature types, positive-class proportion, missing/special values, and justified cleaning. You cannot defend preprocessing choices without looking at the data first.


In [ ]:
df = pd.read_csv(DATA_PATH, sep=";")
print(f"Shape: {df.shape}")
df.head()


In [ ]:
df.info()
df.describe(include="all").T


> **Self-check**
> 1. How many rows and columns did you observe?
> 2. Which columns look numeric vs nominal categorical?
> 3. What is the positive-class base rate, and why does that matter for Average Precision later?


In [ ]:
# Target distribution (positive class = yes)
print(df["y"].value_counts(dropna=False))
print("\nPositive-class proportion (y == 'yes'):", (df["y"] == "yes").mean())


In [ ]:
# Missing values and notable special values
missing = df.isna().sum()
print("Null counts:\n", missing[missing > 0] if missing.any() else "(no pandas-null values)")
print("\npdays value counts (head):")
print(df["pdays"].value_counts().head())
print("\nLiteral 'unknown' counts by column:")
unknown_counts = {c: int((df[c] == "unknown").sum()) for c in df.columns if df[c].dtype == object}
print({k: v for k, v in unknown_counts.items() if v > 0})


> **Learning note — `unknown` vs missing vs unseen categories**
> Literal `"unknown"` is a category value in this dataset, not the same thing as a pandas `NaN`, and not the same as a category that appears only at transform time. Explain your handling in the report.

> **Learning note — `pdays = 999`**
> 999 means *no previous contact*, not 999 elapsed days. Do not silently treat it as a normal numeric distance.


## 2. Target definition and leakage exclusions

- Predict `y`: encode `yes` → 1, `no` → 0.
- **Drop `duration`** from predictors (call duration is unavailable before the call; leakage risk flagged in the dataset docs).
- Document handling of `pdays = 999` (no previous contact) and literal `unknown` category values.


> **Report prompt**
> Write 2–3 sentences for Methods: why `duration` is excluded, and how `pdays=999` / `unknown` are represented.

> **Self-check**
> If you kept `duration`, what would go wrong conceptually at prediction time?


In [ ]:
# Placeholder — implement in Step 1 work:
# y = (df["y"] == "yes").astype(int)
# X = df.drop(columns=["y", "duration"])
pass


## 3. Stratified 70/15/15 train / validation / test split

Use `random_state=42`. Reuse the exact same split rows for every GBDT and MLP experiment. Keep the test set untouched until final model selection is complete.


> **Learning note — why three splits?**
> - Train: fit model + learned preprocessing
> - Validation: early stopping / config selection (validation AP)
> - Test: one-time final evaluation only
> Stratify so each split keeps a similar positive-class rate.

> **Limitation for the Discussion section**
> A random split does **not** prove generalization to future marketing campaigns.


In [ ]:
# Placeholder — implement stratified split after feature matrix is defined.
pass


## 4. Leakage-safe preprocessing

Fit imputation / one-hot encoding (and MLP scaling) on **training data only**, then `transform` validation and test. Keep the same underlying input information and split rows for both models; numeric scaling for MLP is model-specific and should be explained.


> **Learning note — `fit` vs `transform`**
> `fit` / `fit_transform` on train only learns medians, category levels, scaler means, etc. `transform` on val/test applies those learned parameters. Fitting on all data leaks information from held-out rows.

> **TODO (learning)**
> - [ ] Decide `pdays=999` representation
> - [ ] Decide how literal `unknown` is encoded
> - [ ] Build train-fitted ColumnTransformer; scale numerics for MLP
> - [ ] Verify shapes and that `duration` is absent from predictors


In [ ]:
# Placeholder — ColumnTransformer / Pipeline fitted on train only.
pass


## End of Step 0 scaffold

This learning notebook mirrors the Step 0 scaffold (setup, upload, initial load). Placeholder cells are unfinished on purpose. Use `assign2-bank_step1.ipynb` for the completed Step 1 pipeline.
